# NAICS Classification from Pre-computed Embeddings

Using the 3072-dim embeddings (OpenAI text-embedding-3-large) for all 20k companies.

Classifiers: kNN, Logistic Regression, MLP, Random Forest, Linear SVM, LDA, RBF SVM — evaluated at:
- **NAICS-2** (20 sector classes)

In [3]:
import pandas as pd
import numpy as np
import json
import os
import time
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score

print("Imports done.")

Imports done.


In [ ]:
csv_path = '../../.ipynb_checkpoints/ExioNAICS_embeddings_large.csv'
npy_cache = csv_path.rsplit('.', 1)[0] + '_full.npy'

EMB_COLUMNS = ['embeddings']

df = pd.read_csv(csv_path)
print(f"Shape: {df.shape}")

if os.path.exists(npy_cache):
    print(f"\nLoading cached embeddings from {npy_cache}")
    start = time.time()
    X = np.load(npy_cache)
    print(f"Done in {time.time()-start:.1f}s")
else:
    print("\nParsing embeddings...")
    start = time.time()
    arrays = []
    for col in EMB_COLUMNS:
        arr = np.array([json.loads(e) for e in df[col]], dtype=np.float32)
        arrays.append(arr)
    X = np.hstack(arrays)
    print(f"Done in {time.time()-start:.1f}s")
    np.save(npy_cache, X)

df['emb'] = list(X)
print(f"Embedding shape: {X.shape}")

In [ ]:
df = df.drop_duplicates(subset=['Company Name']).reset_index(drop=True)

X = np.array(df['emb'].tolist())
X = normalize(X)

NAICS2_MAP = {31: 33, 32: 33, 44: 45, 48: 49}
df['NAICS_2_merged'] = df['NAICS_2 Code'].replace(NAICS2_MAP)

le2 = LabelEncoder()
y2 = le2.fit_transform(df['NAICS_2_merged'].astype(str))

print(f"After dedup: {len(df)} samples")
print(f"NAICS-2 classes: {len(le2.classes_)}")

print("\nClass distribution:")
counts = pd.Series(y2).value_counts().sort_index()
for cls_idx, count in counts.items():
    print(f"  {le2.classes_[cls_idx]:>5s}: {count:>5d} samples ({count/len(y2):.1%})")

SEED = 192
X_train, X_val, y2_train, y2_val = train_test_split(
    X, y2, test_size=0.1, random_state=SEED
)
print(f"\nTrain: {len(X_train)}, Val: {len(X_val)}")

In [ ]:
TOP_K = [1, 3, 5]
PCA_DIMS = 256

if PCA_DIMS:
    n_features = X_train.shape[1]
    pca = PCA(n_components=PCA_DIMS, random_state=SEED)
    X_train = pca.fit_transform(X_train)
    X_val = pca.transform(X_val)
    print(f"PCA: {n_features} -> {PCA_DIMS} dims ({pca.explained_variance_ratio_.sum():.1%} variance retained)")
else:
    print("PCA: disabled")

def top_k_accuracy(model, X_val, y_val):
    proba = model.predict_proba(X_val)
    results = {}
    for k in TOP_K:
        top_k_preds = np.argsort(proba, axis=1)[:, -k:]
        correct = np.any(top_k_preds == y_val[:, None], axis=1).sum()
        results[f'top{k}'] = correct / len(y_val)
    return results

def format_topk(topk):
    return "  ".join(f"Top-{k}: {topk[f'top{k}']:.4f}" for k in TOP_K)

print(f"Evaluating top-k for k = {TOP_K}")

## 1. kNN Classifier

In [7]:
all_results = {}

for k in [1, 3, 5, 10, 20]:
    knn2 = KNeighborsClassifier(n_neighbors=k, metric='cosine', n_jobs=-1)
    knn2.fit(X_train, y2_train)
    topk2 = top_k_accuracy(knn2, X_val, y2_val)
    all_results[f'knn_k{k}_naics2'] = topk2

    print(f"k={k:<3}  {format_topk(topk2)}")

k=1    Top-1: 0.5375  Top-3: 0.5728  Top-5: 0.5905
k=3    Top-1: 0.5806  Top-3: 0.7447  Top-5: 0.7645
k=5    Top-1: 0.6075  Top-3: 0.7836  Top-5: 0.8260
k=10   Top-1: 0.6245  Top-3: 0.8041  Top-5: 0.8642
k=20   Top-1: 0.6315  Top-3: 0.8267  Top-5: 0.8890


## 2. Logistic Regression

In [ ]:
print("Training Logistic Regression...")
start = time.time()
lr2 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
lr2.fit(X_train, y2_train)
topk2 = top_k_accuracy(lr2, X_val, y2_val)
print(f"  {format_topk(topk2)}  ({time.time()-start:.1f}s)")
all_results['logreg_naics2'] = topk2

## 3. MLP Classifier

In [ ]:
print("Training MLP...")
start = time.time()
mlp2 = MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=200, early_stopping=True, validation_fraction=0.1, random_state=SEED)
mlp2.fit(X_train, y2_train)
topk2 = top_k_accuracy(mlp2, X_val, y2_val)
print(f"  {format_topk(topk2)}  ({time.time()-start:.1f}s)")
all_results['mlp_naics2'] = topk2

## 4. Random Forest

In [ ]:
print("Training Random Forest...")
start = time.time()
rf2 = RandomForestClassifier(n_estimators=100, max_depth=30, max_features='sqrt', class_weight='balanced', random_state=SEED, n_jobs=-1)
rf2.fit(X_train, y2_train)
topk2 = top_k_accuracy(rf2, X_val, y2_val)
print(f"  {format_topk(topk2)}  ({time.time()-start:.1f}s)")
all_results['rf_naics2'] = topk2

## 5. Linear SVM

In [ ]:
print("Training Linear SVM...")
start = time.time()
svm2_base = LinearSVC(C=1.0, max_iter=4000, class_weight='balanced', random_state=SEED)
svm2 = CalibratedClassifierCV(svm2_base, cv=2)
svm2.fit(X_train, y2_train)
topk2 = top_k_accuracy(svm2, X_val, y2_val)
print(f"  {format_topk(topk2)}  ({time.time()-start:.1f}s)")
all_results['svm_naics2'] = topk2

## 6. LDA (Linear Discriminant Analysis)

In [ ]:
print("Training LDA...")
start = time.time()
lda2 = LinearDiscriminantAnalysis()
lda2.fit(X_train, y2_train)
topk2 = top_k_accuracy(lda2, X_val, y2_val)
print(f"  {format_topk(topk2)}  ({time.time()-start:.1f}s)")
all_results['lda_naics2'] = topk2

## 7. RBF SVM

In [ ]:
print("Training RBF SVM...")
start = time.time()
rbf_svm2 = SVC(kernel='rbf', C=10.0, class_weight='balanced', probability=True, random_state=SEED)
rbf_svm2.fit(X_train, y2_train)
topk2 = top_k_accuracy(rbf_svm2, X_val, y2_val)
print(f"  {format_topk(topk2)}  ({time.time()-start:.1f}s)")
all_results['rbf_svm_naics2'] = topk2

## Summary

## 8. Ensemble

In [ ]:
ensemble_models = {
    'logreg': lr2,
    'mlp': mlp2,
    'svm': svm2,
    'lda': lda2,
    'rbf_svm': rbf_svm2,
}

print(f"Ensembling: {', '.join(ensemble_models.keys())}")
avg_proba = np.mean([m.predict_proba(X_val) for m in ensemble_models.values()], axis=0)

topk = {}
for k in TOP_K:
    top_k_preds = np.argsort(avg_proba, axis=1)[:, -k:]
    correct = np.any(top_k_preds == y2_val[:, None], axis=1).sum()
    topk[f'top{k}'] = correct / len(y2_val)

print(f"  {format_topk(topk)}")
all_results['ensemble_naics2'] = topk

In [15]:
header = "  ".join(f"{'Top-'+str(k):>8}" for k in TOP_K)
print("=" * 75)
print("  NAICS-2 RESULTS (20 classes)")
print("=" * 75)
print(f"{'Model':<25} {header}")
print("-" * 75)
for name, topk in all_results.items():
    if 'naics2' in name:
        label = name.replace('_naics2', '')
        vals = "  ".join(f"{topk[f'top{k}']:>8.4f}" for k in TOP_K)
        print(f"{label:<25} {vals}")

# Save results
os.makedirs('results', exist_ok=True)
with open('results/embedding_classifier_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\nResults saved to results/embedding_classifier_results.json")

  NAICS-2 RESULTS (20 classes)
Model                        Top-1     Top-3     Top-5
---------------------------------------------------------------------------
knn_k1                      0.5375    0.5728    0.5905
knn_k3                      0.5806    0.7447    0.7645
knn_k5                      0.6075    0.7836    0.8260
knn_k10                     0.6245    0.8041    0.8642
knn_k20                     0.6315    0.8267    0.8890
logreg                      0.6004    0.8430    0.9180
mlp                         0.6719    0.8727    0.9286
rf                          0.4470    0.7808    0.8607
svm                         0.6711    0.8635    0.9264
lda                         0.6563    0.8635    0.9165
rbf_svm                     0.6895    0.8890    0.9385
ensemble                    0.6719    0.8840    0.9356

Results saved to results/embedding_classifier_results.json
